# Temporal Feature Engineering

**Source:** LinkedIn Learning Course  
**Topic:** Feature Engineering - Temporal and Time Series Techniques  
**Status:** Exploratory notebook for learning temporal feature engineering patterns

## Overview

This notebook demonstrates temporal feature engineering techniques for time series and date-based data in machine learning. It covers extracting datetime components, decomposing time series signals, and building sklearn-compatible transformers for production pipelines.

## Key Techniques Covered

1. **DatetimeFeatures (feature_engine)** - Extract year, month, day, hour, etc. from datetime columns
2. **Seasonal Decomposition** - Break a time series into trend, seasonality, and residual components
3. **Custom `SeasonTransformer`** - Group-wise decomposition wrapped as an sklearn Transformer
4. **Cyclical Encoding** - Encode month/day as sin/cos pairs to capture circular proximity (Dec ↔ Jan)
5. **Linear Trend Index** - Encode elapsed time to capture overall growth/decline
6. **Time-Respecting Train/Test Split** - Never shuffle time series data

## Dataset Context

Uses two datasets:
- **EPA Fuel Economy** (`vehicles.csv`): Demonstrates group-wise seasonal decomposition by car make and end-to-end pipeline with `LinearRegression`
- **Box-Jenkins Airline Passengers** (seaborn `flights`): Classic benchmark time series — monthly international passenger counts 1949–1960. Used for the challenge section.

## Notes for Future Revision

**Security & Data Management:**
- ✅ EPA data loaded from public EPA API (no credentials)
- ✅ Airline data loaded from seaborn's built-in datasets (no credentials)
- ✅ No hardcoded secrets

**Key Learnings:**
- **Cyclical encoding is critical**: Month 12 and Month 1 are adjacent; encoding as raw integers misleads models
- **Temporal splits prevent leakage**: Random shuffling lets future data leak into training
- **Group-wise decomposition**: Each car `make` has its own trend/seasonal — compute per group, not globally
- **`extrapolate_trend='freq'`**: Required by statsmodels to avoid NaN edges after decomposition

**Production Patterns Demonstrated:**
- Wrapping custom logic in `BaseEstimator + TransformerMixin` for pipeline compatibility
- `FunctionTransformer` for inline debug checkpoints inside pipelines
- `ColumnTransformer` composing multiple feature engineering steps

**Related to AI Portfolio:**
- Connects to `notes/01-ml` regression chapters (feature engineering for tabular targets)
- Connects to `notes/02-advanced-deep-learning` sequence chapters (classical vs deep time-series)

**Improvements Needed:**
- [ ] Add lag features (passengers[-1], passengers[-12]) for autoregressive modeling
- [ ] Demonstrate `TimeSeriesSplit` cross-validation instead of a single split
- [ ] Add Fourier features as an alternative to sin/cos encoding
- [ ] Compare seasonal decomposition vs rolling window features
- [ ] Show feature importance for each temporal feature type
- [ ] Add ARIMA baseline for comparison

**Dependencies:**
- pandas, numpy, matplotlib, statsmodels
- scikit-learn, feature_engine
- seaborn (built-in flights dataset)
- category_encoders

**Common Pitfalls to Avoid:**
- ❌ Shuffling time series before train/test split (temporal leakage)
- ❌ Fitting decomposition on entire dataset before split (test data leakage)
- ❌ Encoding month as a raw integer (Dec=12, Jan=1 appear far apart)

---

### Loading and Parsing the Dataset
This block downloads fuel economy data from the web and cleans the timezone information.

It features a custom function `to_tz` to convert naive string dates into timezone-aware datetimes.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

import pandas as pd
import numpy as np
from utils import (
    load_fuel_economy_data,
    debug_transformer,
    TextPipeline,
    CAT_COLS, LOW_CARDINALITY_COLS, HIGH_CARDINALITY_COLS,
    make_preprocessor,
)

### Data Preview
Let's peek at the first few rows of the cleaned fuel economy dataset.


In [ ]:
# analyzing the data
df = load_fuel_economy_data()

df.head()

### Install feature_engine
Installing the `feature_engine` package for datetime feature engineering.


In [ ]:
!pip install feature_engine

### Extract Temporal Features
Using `feature_engine.datetime.DatetimeFeatures` to extract elements like year, month, day, hour, etc. directly from the datetime column.


In [ ]:
from feature_engine.datetime import DatetimeFeatures
dtf = DatetimeFeatures(features_to_extract='all')
dtf.fit_transform(df[['createdOn']])

### Install statsmodels
Installing `statsmodels` to access its advanced time series analysis tools.


In [ ]:
!pip install statsmodels

### Documentation Check
Checking the docstring for `seasonal_decompose`.


In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

seasonal_decompose?

### Data Prep: Filtering and Forward Fill
We will extract data specifically for Ford vehicles to demonstrate time-series decomposition.

We group by year, take the median `city08` MPG, and use forward fill (`.ffill()`) to handle missing years.


In [ ]:
"""
📊 FORWARD FILL (ffill) DIAGRAM
If a year has no data, fill it with the previous year's value to keep the time-series continuous.
1999: 15 mpg
2000: NaN -> becomes 15 mpg
2001: 16 mpg
"""
ford = (
    df.query("make=='Ford'") # 1. Filter for Ford
    .groupby('year')         # 2. Group by Year
    .city08                  # 3. Select the city08 (city MPG) column
    .median()                # 4. Take the median for each year
    .ffill()                 # 5. Forward fill any missing years
)

ford.plot(title='Ford City MPG over Time')

### Train / Test Split
Preparing the dataset for Machine Learning by separating features (`X`) from the target variable (`y`).


In [ ]:
from sklearn.model_selection import train_test_split

# create X (features) and y (target)
# X gets all columns EXCEPT the target
X = df.drop(columns=['city08'])
# y gets ONLY the target
y = df.city08

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

### Visualizing Decomposition
Here we visually break down the Ford time series into its core components: Original, Trend, Seasonality, and Residuals.


In [ ]:
from matplotlib import pyplot as plt
decomposition = seasonal_decompose(ford, model='additive', period=5)

# plot the decomposition
# creates a fig with 4 rows and one column of subplots, ax corresponding to each subplot
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(15, 10))

# Plot the Original Data
ford.plot(ax=ax1, title='Original')

# "Decomposes" (slices/aggregates) the time series data into different views
# Plot the Trend (overall direction)
decomposition.trend.plot(ax=ax2, title='Trend')

# Plot the Seasonality (short-term cycle)
decomposition.seasonal.plot(ax=ax3, title='Seasonality')

# Plot the Residuals (noise)
decomposition.resid.plot(ax=ax4, title='Residuals')
fig.tight_layout()

### Advanced Decomposition Orchestrator
The functions below automatically compute the trend, seasonality, and residuals for *every single car make* independently, and then stitches everything back into the original dataset order.

*(See the inline Python comments and diagrams for the full flow.)*


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose

def get_seasonal(group, time_col, agg_col, period=1):
    """
    This function acts as the "Worker". It runs individually on each separate
    group (e.g., just the Ford rows, then just the Honda rows).
    """

    # 1. Sort the group chronologically (e.g., from year 1990 to 2020)
    group = group.sort_values(time_col)

    # 2. Save the original DataFrame index so we don't lose track of which row is which
    group.index.name = 'index'

    # 3. Create a clean Time Series (ts) where the index is the Time and the value is the Metric
    ts = group.set_index(time_col)[agg_col]

    # 4. Math Check: We can't find a trend if there's only 1 data point!
    if len(ts) < 2:
        return group.assign(seasonal=0, trend=0, resid=0)

    # 5. Do the complex math (Statsmodels) to break time series into trend, seasonal, and residual.
    res = seasonal_decompose(ts, model='additive', period=period, extrapolate_trend='freq')

    # 6. Take the math results and staple them as new columns onto our group using .assign()
    #
    # THE "MATH STAPLER" DIAGRAM
    #        group (Ford only)                     res (Decomposition Results)
    # +-------+------+------+---------+       +----------+-------+-------+
    # | index | year | make | barrels |       | seasonal | trend | resid |
    # +-------+------+------+---------+       +----------+-------+-------+
    # |   0   | 2000 | Ford |   15    |   +   |   0.5    |  14   |  0.5  |
    # |   2   | 2001 | Ford |   14    |       |   0.2    |  13   |  0.8  |
    # +-------+------+------+---------+       +----------+-------+-------+
    #                                 |
    #                                 V  .assign()
    #                                 |
    # +-------+------+------+---------+----------+-------+-------+
    # | index | year | make | barrels | seasonal | trend | resid |
    # +-------+------+------+---------+----------+-------+-------+
    # |   0   | 2000 | Ford |   15    |   0.5    |  14   |  0.5  |
    # |   2   | 2001 | Ford |   14    |   0.2    |  13   |  0.8  |
    # +-------+------+------+---------+----------+-------+-------+
    return (group
            .assign(seasonal=res.seasonal.values,
                    trend=res.trend.values,
                    resid=res.resid.values))

def add_seasonal(df, time_col, group_cols, agg_col):
    """
    This function is the "Orchestrator". It splits the massive dataset into smaller,
    manageable chunks (grouped by the car's 'make'), processes them, and glues them back together.

    THE GROUPBY FLOW DIAGRAM
    [ ENTIRE DATASET ]
      Index | Year | Make   | Barrels
        0   | 2000 | Ford   |   15
        1   | 2005 | Honda  |   10
        2   | 2001 | Ford   |   14
            |
            V  df.groupby(['make'])
            |
    +-------------------+      +-------------------+
    |   GROUP: Ford     |      |   GROUP: Honda    |
    | 0 | 2000 | Ford   |      | 1 | 2005 | Honda  |
    | 2 | 2001 | Ford   |      +-------------------+
    +-------------------+               |
            |                           |
            V  .apply(get_seasonal)     V
            |                           |
    [ Calculates Trend  ]      [ Calculates Trend  ]
    [ for Ford over time]      [ for Honda over time]
            |                           |
            +-------------+-------------+
                          |
                          V
            [ RECOMBINED ENTIRE DATASET ]
    """

    return (df
            .groupby(group_cols)                                                          # 1. Split data into chunks based on 'make'
            .apply(get_seasonal, time_col=time_col, agg_col=agg_col, period=1,           # 2. Process each chunk
                   include_groups=False)                                                  #    include_groups=False: pandas 2.2+ avoids FutureWarning;
                                                                                          #    group key ('make') goes into MultiIndex, not the group DataFrame
            .reset_index(drop=False)                                                      # 3. MultiIndex ('make', 'index') -> columns 'make' and 'index'
            .set_index('index')                                                           # 4. Restore the original row index
            .loc[df.index])                                                               # 5. Re-sort rows to match the exact original order

# Note: X needs to be defined earlier in the notebook for this to work
res = add_seasonal(X, 'year', ['make'], agg_col='barrels08')

### View Results
Let's peek at the final few columns of our newly generated dataframe containing the decomposition results.


In [ ]:
res.iloc[:, -5:]

### Train / Test Split
Preparing the dataset for Machine Learning by separating features (`X`) from the target variable (`y`).


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# create X and y (exclude all target-leaking columns plus createdOn)
X = df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y = df.city08

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


# SeasonTransformer wraps add_seasonal so it can sit in a sklearn Pipeline
class SeasonTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, time_col, group_cols, agg_col):
        self.time_col = time_col
        self.group_cols = group_cols
        self.agg_col = agg_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return add_seasonal(X, self.time_col, self.group_cols, self.agg_col)


# Full pipeline: seasonal features -> preprocessing -> debug -> regression
pipeline = Pipeline(steps=[
    ('seasonal_decompose', SeasonTransformer(time_col='year', group_cols=['make'], agg_col='barrels08')),
    ('preprocessor', make_preprocessor()),
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name': 'tmp_X'})),
    ('lr', LinearRegression()),
])

pipeline.fit(X_train, y_train)

### Model Evaluation
Calculate the $R^2$ score on the test dataset.


In [ ]:
pipeline.score(X_test, y_test)

**Challenge**

Predict air passenger volume

In [ ]:
# seaborn provides the classic airline passengers dataset used in the challenge below
!pip install seaborn --quiet

In [ ]:
import seaborn as sns

# Load the classic Box-Jenkins international airline passengers dataset (1949-1960)
# 12 months × 12 years = 144 monthly observations of total passenger counts
raw_data = sns.load_dataset('flights')

# Reformat to a Year/Month integer structure compatible with tweak_airline below
_month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
              'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
raw = (raw_data
       .rename(columns={'year': 'Year', 'passengers': 'Passengers'})
       .assign(Month=raw_data['month'].map(_month_map))
       .drop(columns=['month'])
      )

In [ ]:
raw

In [ ]:
def tweak_airline(df_):
    # seaborn's flights dataset has integer Month values; no TOTAL rows to filter
    return (df_
            .astype({'Month': 'int64'})
            .assign(date=lambda df: pd.to_datetime(df[['Year', 'Month']].assign(day=1)))
    )

air = tweak_airline(raw)
air

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
from matplotlib import pyplot as plt

# --- Temporal Feature Engineering for Airline Passengers ---
#
# CYCLICAL ENCODING DIAGRAM
# Raw month:  1  2  3  4  5  6  7  8  9 10 11 12  1  2  ...
# Problem:    model sees 12 and 1 as 11 steps apart (they should be adjacent!)
# Solution:   encode as (sin, cos) on a unit circle — Dec and Jan are geometrically adjacent
#
#   month_sin = sin(2π × month / 12)
#   month_cos = cos(2π × month / 12)

air_feats = (air
    .assign(
        year=air.date.dt.year,
        month=air.date.dt.month,
        month_sin=lambda df: np.sin(2 * np.pi * df.month / 12),
        month_cos=lambda df: np.cos(2 * np.pi * df.month / 12),
        # Linear trend index: captures overall growth over the series
        # t=0 is Jan 1949, t=143 is Dec 1960
        t=range(len(air)),
    )
)

X_air = air_feats[['year', 'month', 'month_sin', 'month_cos', 't']]
y_air = air_feats['Passengers']

# Time-respecting train/test split — NEVER shuffle time series data
# Shuffling would allow future months to inform predictions of past months (leakage)
split = int(len(X_air) * 0.8)
X_train_air, X_test_air = X_air.iloc[:split], X_air.iloc[split:]
y_train_air, y_test_air = y_air.iloc[:split], y_air.iloc[split:]

# Pipeline: scale features then fit a gradient boosting regressor
air_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                        learning_rate=0.1, random_state=42))
])

air_pipeline.fit(X_train_air, y_train_air)
print(f"R² on test set: {air_pipeline.score(X_test_air, y_test_air):.3f}")

# Visualise actual vs predicted over the full timeline
y_pred_all = air_pipeline.predict(X_air)

fig, ax = plt.subplots(figsize=(12, 4))
y_air.reset_index(drop=True).plot(ax=ax, label='Actual', color='steelblue', linewidth=2)
pd.Series(y_pred_all, name='Predicted').plot(
    ax=ax, label='Predicted', color='darkorange', linestyle='--', linewidth=2)
ax.axvline(split, color='crimson', linestyle=':', linewidth=1.5,
           label=f'Train / Test split (month {split})')
ax.set_title('International Airline Passengers: Actual vs Predicted')
ax.set_xlabel('Months since Jan 1949')
ax.set_ylabel('Passengers (thousands)')
ax.legend()
plt.tight_layout()
plt.show()